### 04 —  MLflow + MinIO 

In [ ]:
import os, sys, time, glob, pathlib, subprocess, urllib.request

os.chdir("/kaggle/working")
REPO = "/kaggle/working/repo"
subprocess.run(["rm","-rf",REPO])
subprocess.run(["git","clone","-b","project-defense",
    "https://github.com/FilthyName/Processing_and_analysis_of_medical_images.git", REPO], check=True)
os.chdir(REPO); sys.path.append(f"{REPO}/src")

META = glob.glob("/kaggle/input/**/HAM10000_metadata.csv", recursive=True)[0]
IMAGES_DIR = "/kaggle/working/images"; pathlib.Path(IMAGES_DIR).mkdir(exist_ok=True)
for p in glob.glob("/kaggle/input/**/*.jpg", recursive=True):
    link = os.path.join(IMAGES_DIR, os.path.basename(p))
    if not os.path.exists(link): os.symlink(p, link)
print("картинок:", len(glob.glob(f"{IMAGES_DIR}/*.jpg")))

subprocess.run(["python","src/make_splits.py","--metadata",META,
    "--images-dir",IMAGES_DIR,"--out-dir","data/splits","--seed","42"], check=True)

subprocess.run(["pip","install","-q","mlflow==2.16.2","boto3","hydra-core","omegaconf"], check=True)

os.makedirs("/kaggle/working/minio-data", exist_ok=True)
for name,url in [("minio","https://dl.min.io/server/minio/release/linux-amd64/minio"),
                 ("mc","https://dl.min.io/client/mc/release/linux-amd64/mc")]:
    f=f"/kaggle/working/{name}"
    if not os.path.exists(f): urllib.request.urlretrieve(url,f); os.chmod(f,0o755)

os.environ.update(MINIO_ROOT_USER="minioadmin", MINIO_ROOT_PASSWORD="minioadmin")
subprocess.Popen(["/kaggle/working/minio","server","/kaggle/working/minio-data",
    "--address",":9000","--console-address",":9001"],
    stdout=open("/kaggle/working/minio.log","w"), stderr=subprocess.STDOUT)
time.sleep(8)
subprocess.run(["/kaggle/working/mc","alias","set","local","http://localhost:9000","minioadmin","minioadmin"])
subprocess.run(["/kaggle/working/mc","mb","-p","local/mlflow"])

os.environ.update(
    MLFLOW_S3_ENDPOINT_URL="http://localhost:9000",
    AWS_ACCESS_KEY_ID="minioadmin", AWS_SECRET_ACCESS_KEY="minioadmin",
    MLFLOW_TRACKING_URI="http://localhost:5000")

subprocess.Popen(["mlflow","server",
    "--backend-store-uri","sqlite:////kaggle/working/mlflow.db",
    "--artifacts-destination","s3://mlflow/","--serve-artifacts",
    "--host","0.0.0.0","--port","5000"],
    env=os.environ.copy(), stdout=open("/kaggle/working/mlflow.log","w"), stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        if urllib.request.urlopen("http://localhost:5000/health").read(): break
    except Exception: time.sleep(2)
print("MLflow:", urllib.request.urlopen("http://localhost:5000/health").read())

In [ ]:
!cd /kaggle/working/repo && python src/dl_experiments.py \
  data.images_dir=/kaggle/working/images \
  data.splits_dir=/kaggle/working/repo/data/splits

In [ ]:
import subprocess
subprocess.run(["pip","install","-q","mlflow==2.16.2","boto3"], check=True)

import os
os.environ.update(MLFLOW_TRACKING_URI="http://localhost:5000",
                  MLFLOW_S3_ENDPOINT_URL="http://localhost:9000",
                  AWS_ACCESS_KEY_ID="minioadmin", AWS_SECRET_ACCESS_KEY="minioadmin")
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")
c = mlflow.tracking.MlflowClient()

print("=== PRD-модель ===")
for mv in c.search_model_versions("name='skin_lesion_vit_b16'"):
    print(f"  версия {mv.version} | теги: {mv.tags}")

print("\n=== Раны и метрики ===")
exp = c.get_experiment_by_name("skin_lesion_isic2018")
for r in c.search_runs(exp.experiment_id):
    m = r.data.metrics
    print(f"  {r.data.tags.get('mlflow.runName')}: "
          f"test_macro_f1={round(m.get('test_macro_f1',0),4)}, "
          f"mel_sens_thr={round(m.get('test_mel_sensitivity_thr',0),4)}")

In [ ]:
!cd /kaggle/working/repo && python src/dl_demonstration.py \
  data.images_dir=/kaggle/working/images \
  data.splits_dir=/kaggle/working/repo/data/splits

In [ ]:
import os
for f in ["/kaggle/working/checkpoint7_results.zip", "/kaggle/working/ckpt7_core.zip"]:
    if os.path.exists(f): os.remove(f)
# посмотреть, сколько занято
!df -h /kaggle/working
!du -sh /kaggle/working/* 2>/dev/null | sort -h | tail -10

In [ ]:
import subprocess, os
subprocess.run(
  "cd /kaggle/working && zip -r ckpt7_light.zip mlflow.db repo/outputs 2>/dev/null",
  shell=True)
print("ckpt7_light.zip:", round(os.path.getsize('/kaggle/working/ckpt7_light.zip')/1e6,2), "МБ")

In [ ]:
import os, glob, shutil, subprocess
os.environ.update(MLFLOW_TRACKING_URI="http://localhost:5000",
                  MLFLOW_S3_ENDPOINT_URL="http://localhost:9000",
                  AWS_ACCESS_KEY_ID="minioadmin", AWS_SECRET_ACCESS_KEY="minioadmin")
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")
c = mlflow.tracking.MlflowClient()
exp = c.get_experiment_by_name("skin_lesion_isic2018")
run = c.search_runs(exp.experiment_id, filter_string="tags.mlflow.runName='vit_b16_final'")[0]

# (A) модель для разраба
m = mlflow.artifacts.download_artifacts(run_id=run.info.run_id,
        artifact_path="service_checkpoint", dst_path="/kaggle/working/_dl")
shutil.copy(glob.glob(m+"/*.pth")[0], "/kaggle/working/model_prd.pth")
print("МОДЕЛЬ:", round(os.path.getsize('/kaggle/working/model_prd.pth')/1e6,1), "МБ")

# (B) графики для презы — стащим все png из артефактов рана
allart = mlflow.artifacts.download_artifacts(run_id=run.info.run_id, dst_path="/kaggle/working/_art")
os.makedirs("/kaggle/working/plots_for_slides", exist_ok=True)
for p in glob.glob(allart+"/**/*.png", recursive=True):
    shutil.copy(p, "/kaggle/working/plots_for_slides/")
for p in glob.glob(allart+"/**/*.csv", recursive=True):
    shutil.copy(p, "/kaggle/working/plots_for_slides/")
print("ГРАФИКИ:", os.listdir("/kaggle/working/plots_for_slides"))

# (C) база MLflow для скриншотов UI
subprocess.run("cd /kaggle/working && zip -q plots_and_db.zip mlflow.db -r plots_for_slides", shell=True)
print("готово: model_prd.pth + plots_and_db.zip")

In [ ]:
import os, subprocess

src = "/kaggle/working/model_prd.pth"
print("есть файл:", os.path.exists(src), round(os.path.getsize(src)/1e6,1), "МБ")

# архивируем во /tmp (другой диск), чтобы не упереться в место, потом переносим
subprocess.run("cd /kaggle/working && zip -1 /tmp/model_prd.zip model_prd.pth && mv /tmp/model_prd.zip .", shell=True)
print("архив:", round(os.path.getsize('/kaggle/working/model_prd.zip')/1e6,1), "МБ")